# Batch processing of 800 (many) baselines to produce single power spectra

## IMPORTS

In [ ]:
import os
import toml
import json
import papermill as pm
import numpy as np
from pyuvdata import UVData
from hera_filters import dspec

## CONSTANTS

In [ ]:
#-------GLOBAL CONSTANTS-------#
C_M_PER_NS = 0.299792458

#------Eigenvalue cutoff search settings--------#
T_LO, T_HI = - 14.0, 12.0
T_TOL = 1e-3

#------Filter half-width limits------#
HW_FRACT_LO = 0.1
HW_FRACT_HI = 1.0

#------------LOADING TOML CONFIGS------------#

TOML_FILE = os.environ.get('TOML_FILE',
        '/home/Kwuzard/Projects/HERA_HONS/src/HONS/h6c_pspec_11band.toml')

# Load configuration from the toml and inject into globals.
toml_options = toml.load(TOML_FILE)
for toml_section in ['GLOBAL_OPTS', 'POSTPROCESS_AND_PSPEC_OPTS']:
    if toml_section not in toml_options:
        continue
    print(f'\nLoading config from [{toml_section}] in {TOML_FILE}:')
    for key, val in toml_options[toml_section].items():
        globals()[key.upper()] = val
        print(f'  {key.upper()} = {val!r}')

In [ ]:
BATCH_DATA_DIR = None
BATCH_OUT_DIR = None

## Helpers

In [ ]:
def _pair_name(input_file):
    return input_file.name.split(".")[3]

def n_modes_kept(freqs_hz, filter_hw_ns, cutoff, cache=None):
    """Number of DPSS modes the filter actually fits (columns of the design matrix)."""
    _, nterms = dspec.dpss_operator(
        freqs_hz, [0.0], [filter_hw_ns * 1e-9], cache=cache, eigenval_cutoff=[cutoff]
    )
    return int(nterms[0])

def _t_to_cutoff(t):
    c = 10.0 ** t / (1.0 + 10.0 ** t)
    return float(f"{c:.8g}")

def find_cutoff_for_n_modes(n_modes_fn, target_k, t_lo=-14.0, t_hi=5.0, t_tol=1e-3):
    """
    Invert cutoff -> n_modes without needing the eigenvalues directly.
    n_modes_fn(cutoff) is non-increasing in the cutoff, so we bisect on
    t = log10(c / (1 - c)) (same axis as the bisection search) to find the t-interval
    over which exactly `target_k` modes are kept, and return the cutoff in the middle
    of that interval (the middle of the eigenvalue gap -> robust to float handling).

    Returns (cutoff, achieved_k, status), status in {"ok", "k_exceeds_available",
    "cannot_hit_k"}. cutoff is None unless status == "ok".
    """
    def n_at(t):
        try:
            return int(n_modes_fn(_t_to_cutoff(t)))
        except Exception:
            return 0

    if target_k < 1:
        return None, 0, "cannot_hit_k"
    if n_at(t_lo) < target_k:
        return None, n_at(t_lo), "k_exceeds_available"

    def last_t_with_at_least(k):
        """Largest t where n_modes >= k (t_lo - tiny if even t_lo has fewer)."""
        if n_at(t_hi) >= k:
            return t_hi
        if n_at(t_lo) < k:
            return t_lo
        lo, hi = t_lo, t_hi
        while hi - lo > t_tol:
            mid = 0.5 * (lo + hi)
            if n_at(mid) >= k:
                lo = mid
            else:
                hi = mid
        return lo

    t_a = last_t_with_at_least(target_k)        # upper end of the "K modes" interval
    t_b = last_t_with_at_least(target_k + 1)    # lower end (K+1 modes below this)
    for t in (0.5 * (t_a + t_b), t_a):
        if n_at(t) == target_k:
            return _t_to_cutoff(t), target_k, "ok"
    return None, n_at(t_a), "cannot_hit_k"

def _read_horizon(folder):
    """Read the baseline horizon (ns) from hw.csv, or None if missing."""
    hw_csv = folder / "hw.csv"
    if not hw_csv.is_file():
        return None
    with open(hw_csv) as f:
        return float(f.readline().strip().split(",")[1])

def get_baseline_horizon(file_to_use, horizon_folder):
    """Run the notebook once at 1.0 x horizon (unless cached) and return the horizon."""
    HORIZON_HW_FRACT = 1.0        # the horizon is defined by the 1.0 x horizon run only
    HORIZON_EV_CUTOFF = 1e-12     # any cutoff works; hw.csv does not depend on it

    horizon = _read_horizon(horizon_folder)
    if horizon is not None:
        return horizon
 
    horizon_folder.mkdir(parents=True, exist_ok=True)
    try:
        pm.execute_notebook(
            SINGLE_BL_PSPEC_NOTEBOOK_PATH,
            horizon_folder / "sbpp_output.ipynb",
            parameters={
                "INPUT_FILE": str(file_to_use),
                "OUTPUT_DIR": str(horizon_folder.absolute()),
                "PERFORM_DLY_FILT": True,
                "HW_OUT_DIR": str(horizon_folder.absolute()),
                "DLY_FILT_HW_FRACT": HORIZON_HW_FRACT,
                "PERFORM_INPAINT": False,
                "DLY_FILT_EIGENVAL_CUTOFF": HORIZON_EV_CUTOFF,
                "TERMINATE_BEFORE_FILTERING": True,
            },
        )
    except Exception as e:
        print(f"Horizon notebook failed for {file_to_use.name}: {type(e).__name__}: {e}")
 
    return _read_horizon(horizon_folder)


## Batch processing

In [ ]:
if not BATCH_DATA_DIR:
    print(f"Path to batch data is invalid : {BATCH_DATA_DIR}")
    print(f"Please set this path to a directory containing all the redundantly averaged visibility datasets.")
    os._exit(00)

if not BATCH_OUT_DIR:
    print(f"Path to batch output is invalid : {BATCH_OUT_DIR}")
    print(f"Please set this path to a directory where you would like all the power spectra to go.")
    os._exit(00)

#Create directories
# ---- Output folders ----
pspecs_and_horizons = BATCH_OUT_DIR / "pspecs_and_horizons"
horizons_folder = pspecs_and_horizons / "horizons"
pspecs_unfiltered_folder = pspecs_and_horizons / "pspec_out" / "unfiltered"
pspecs_filtered_folder = pspecs_and_horizons / "pspec_out" / "filtered"

pspecs_and_horizons.mkdir(parents = True, exist_ok = True)
horizons_folder.mkdir(parents = True, exist_ok = True)
pspecs_unfiltered_folder.mkdir(parents = True, exist_ok = True)
pspecs_filtered_folder.mkdir(parents = True, exist_ok = True)

#Identify the fit result path
fit_path = None
if FIT_ON_EXPERIMENTAL:
    fit_path = FIT_RESULTS_OUT / "fit_testing" / "fit_results" / "fit_result.json"
else:
    if FIT_ON_EOR_ONLY:
        fit_path = FIT_RESULTS_OUT / "eor_only" / "fit_testing" / "fit_results" / "fit_result.json"
    else:
        fit_path = FIT_RESULTS_OUT / "eor_and_foregrounds" / "fit_testing" / "fit_results" / "fit_result.json"

#Try to extract fits
m_hw, c_hw = (0, 0)
m_n, c_n = (0, 0)

try:
    with open(str(fit_path), "r") as f:
        loaded_fit_data = json.load(f)

    m_hw, c_hw = loaded_fit_data["chosen_filter_hw_fit"]
    m_n, c_n = loaded_fit_data["normalized_n_modes_fit"]

except Exception as e:
    raise RuntimeError(f"Failed to load fit data from {fit_path}") from e

input_files = []
for input_file in sorted(BATCH_DATA_DIR.glob("*.uvh5")):
    ant1, ant2 = _pair_name(input_file).split("_")
    if ant1 != ant2:
        input_files.append(input_file)

#Produce power spectra for all baselines based off their optimal configuration fit value (for filtered pspecs)
for input_file in input_files:
    bl_name = "BLPAIR_" + _pair_name(input_file)

    bl_pspec_filtered_dir = pspecs_filtered_folder / bl_name
    bl_pspec_unfiltered_dir = pspecs_unfiltered_folder / bl_name
    bl_horizon_folder = horizons_folder / bl_name

    bl_horizon_folder.mkdir(parents = True, exist_ok = True)
    bl_pspec_filtered_dir.mkdir(parents = True, exist_ok = True)
    bl_pspec_unfiltered_dir.mkdir(parents = True, exist_ok = True)

    #Determine horizon
    horizon = get_baseline_horizon(input_file, bl_horizon_folder)
    if horizon is None:
        print(f"Skipping {bl_name}: could not determine horizon.")
        continue

    #Determine bandwidth
    uvd = UVData()
    uvd.read(input_file, read_data=False)
    freqs_hz = np.squeeze(uvd.freq_array)
    bandwidth = freqs_hz.max() - freqs_hz.min() + np.median(np.diff(freqs_hz))

    #Determine fitted values
    bl_len = horizon * C_M_PER_NS
    num_modes_normalized = m_n * bl_len + c_n
    filter_hw_ns = m_hw * bl_len + c_hw
    hw_fract =  max(HW_FRACT_LO, min(filter_hw_ns / horizon, HW_FRACT_HI))
    two_NW = 2.0 * bandwidth * filter_hw_ns * 1e-9
    num_modes = int(round(num_modes_normalized * two_NW))

    if num_modes < 1 or filter_hw_ns <= 0:
        print(f"Invalid configuration (hw_ns: {filter_hw_ns}, num_modes: {num_modes})")

    #Determine the eigenvalue cutoff via inverse search
    ev_cutoff, achieved_k, map_status = find_cutoff_for_n_modes(
        lambda c: n_modes_kept(freqs_hz, filter_hw_ns, c, None),
        target_k = num_modes,
        t_lo=  T_LO, t_hi = T_HI, t_tol = T_TOL,
    )

    if map_status != "ok":
        print(f"Could not determine eigenvalue cutoff for {bl_name} with predicted config (hw: {filter_hw_ns}, n_modes: {num_modes})")
        print(f"Skipping {bl_name}")
        continue

    #Produce the unfiltered power spectrum
    unfiltered_pspec = bl_pspec_unfiltered_dir / f"{input_file.stem}.tavg.pspec.h5"
    if not unfiltered_pspec.is_file():
        print(f"Processing file without delay filter: {input_file.name}")
        try:
            pm.execute_notebook(
                SINGLE_BL_PSPEC_NOTEBOOK_PATH,
                bl_pspec_unfiltered_dir / "sbpp_output.ipynb",
                parameters={
                    "INPUT_FILE": str(input_file),
                    "OUTPUT_DIR": str(bl_pspec_unfiltered_dir.absolute()),
                    "PERFORM_DLY_FILT": False,
                    "PERFORM_INPAINT": False,
                },
            )
        except Exception as e:
            print(f"Notebook failed for unfiltered PSPEC {input_file.name}: "
                    f"{type(e).__name__}: {e}")
    if not unfiltered_pspec.is_file():
        print(f"Skipping {bl_name}: unfiltered power spectrum was not created.")
        continue


    #Produce the filtered power spectrum
    filtered_pspec = bl_pspec_filtered_dir / f"{input_file.stem}.tavg.pspec.h5"
    if not filtered_pspec.is_file():
        print(f"Processing file with delay filter: {input_file.name}")
        try:
            pm.execute_notebook(
                SINGLE_BL_PSPEC_NOTEBOOK_PATH,
                bl_pspec_filtered_dir / "sbpp_output.ipynb",
                parameters={
                    "INPUT_FILE": str(input_file),
                    "OUTPUT_DIR": str(bl_pspec_filtered_dir.absolute()),
                    "PERFORM_DLY_FILT": True,
                    "HW_OUT_DIR": str(bl_pspec_filtered_dir.absolute()),
                    "DLY_FILT_HW_FRACT": hw_fract,
                    "PERFORM_INPAINT": False,
                    "DLY_FILT_EIGENVAL_CUTOFF": ev_cutoff,
                },
            )
        except Exception as e:
            print(f"Notebook failed for filtered PSPEC with HW fraction={hw_fract}, "
                    f"EV cutoff={ev_cutoff}: {type(e).__name__}: {e}")
    if not filtered_pspec.is_file():
        print(f"Skipping {bl_name}: filtered power spectrum was not created.")
        continue